In [ ]:
import numpy as np
from sklearn.neural_network import BernoulliRBM

In [ ]:
# Exercise settings

## params 
### params from exceris 
Y = np.array([
    [0, 1, 1],
    [1, 0, 1],
    [0, 1, 1],
    [1, 0, 0],
    [1, 0, 0]
])

m = 5
n = 3
beta = 1.0

### Edited params for model
W_custom = beta * (2 * Y - 1)
c_custom = beta * np.sum(Y, axis=0)      
b_custom = beta * np.zeros(m)

# Model
## Initialization
rbm = BernoulliRBM(n_components=n, random_state=42)
## Structure initialization
fake_data = np.zeros((1, m))
rbm.partial_fit(fake_data)
## Put given params
rbm.components_ = W_custom.T.copy()
rbm.intercept_hidden_ = c_custom.copy()
rbm.intercept_visible_ = b_custom.copy()

In [ ]:
v_sample = np.array([[0, 1, 1, 0, 1]]) # pierwszy wiersz Y
p_h_given_v = rbm.transform(v_sample)
print("Prawdopodobieństwa H:", p_h_given_v)

In [ ]:
import numpy as np
from collections import Counter
import itertools

# ==========================================
# Measurement Methods
# ==========================================

### Helper method: Generate all combinations of the configuration space (V, H)
# In total we have m=5 and n=3, which equals 5 + 3 = 8 bits. 2^8 = 256 states[cite: 4].
# Note: The assignment text mentions 328 configurations (likely a typo in the PDF, as 2^8 = 256)[cite: 17].
def generate_all_configurations(m, n):
    """
    Generates all possible binary combinations for visible (V) and hidden (H) layers[cite: 4].
    """
    # Generate all binary combinations for V and H independently [cite: 4]
    all_v = np.array(list(itertools.product([0, 1], repeat=m)))
    all_h = np.array(list(itertools.product([0, 1], repeat=n)))
    return all_v, all_h


### Measurement method for Task 1b: Analytical energy and probability evaluation
def measure_exact_energy_and_probabilities(rbm, m, n):
    """
    Calculates the exact energy and exact Gibbs distribution probability for every single state[cite: 5, 6, 8, 20].
    """
    all_v, all_h = generate_all_configurations(m, n)
    
    # Extract learned/assigned parameters from the scikit-learn model
    W = rbm.components_.T
    c = rbm.intercept_hidden_
    b = rbm.intercept_visible_
    
    total_unnormalized_prob = 0
    configs = []
    
    # 1. Compute unnormalized energy for each (v, h) combination [cite: 8]
    for v in all_v:
        for h in all_h:
            # Energy formula from the document: E = -v^T * W * h - c^T * h - b^T * v [cite: 8]
            energy = -np.dot(v, np.dot(W, h)) - np.dot(c, h) - np.dot(b, v)
            
            # Unnormalized probability assuming beta=1 is embedded in the parameters [cite: 6]
            unnormalized_prob = np.exp(-energy) 
            
            configs.append({
                'v': v, 
                'h': h, 
                'energy': energy, 
                'unnorm_p': unnormalized_prob
            })
            total_unnormalized_prob += unnormalized_prob
            
    # 2. Normalization – computing the real probabilities using the partition function [cite: 5, 6]
    for cfg in configs:
        cfg['probability'] = cfg['unnorm_p'] / total_unnormalized_prob
        del cfg['unnorm_p'] # Clean up temporary unnormalized values
        
    # Sort configurations from lowest energy to highest (highest probability to lowest) [cite: 18]
    configs.sort(key=lambda x: x['energy'])
    
    # Print results for the top 5 states with the minimum energy [cite: 18, 20]
    print("\n--- [Task 1b] Numerical/Analytical Results (Top 5 states with minimum energy): ---")
    for i in range(5):
        print(f"Rank {i+1}: V={configs[i]['v']}, H={configs[i]['h']} | Energy: {configs[i]['energy']:.2f} | Exact Probability: {configs[i]['probability']:.6f}")
        
    return configs


### Measurement method for Task 1a: Stochastic simulation using Gibbs Sampler
def measure_gibbs_simulation(rbm, m, steps=100000):
    """
    Runs the Gibbs sampling chain and estimates the stochastical probabilities of all visited configurations[cite: 12, 13, 14, 17].
    """
    # Initialize the counter to track the occurrences of each (v, h) configuration
    config_counter = Counter()
    
    # Start with a random binary visible state vector V [cite: 4]
    v_current = np.random.randint(0, 2, size=(1, m)).astype(np.float32)
    
    print(f"\n--- Running Gibbs Sampler ({steps} steps)... ---") [cite: 17]
    for _ in range(steps):
        # 1. Gibbs sampling step using scikit-learn's built-in mechanism: V_old -> H -> V_new [cite: 13]
        v_next = rbm.gibbs(v_current)
        
        # 2. Reconstruction of the hidden layer H state corresponding to this step [cite: 10, 13]
        # Calculate conditional probabilities p(H=1|V) [cite: 10]
        p_h = rbm._sample_h(v_next) 
        
        # Perform binary stochastic sampling for the hidden state vector H [cite: 10, 11]
        h_sampled = (np.random.rand(*p_h.shape) < p_h).astype(int)[0]
        
        # Flatten and convert the next visible state to clean integers
        v_flat = v_next[0].astype(int)
        
        # 3. Create a unique dictionary key as a tuple of tuples: (tuple_V, tuple_H)
        state_key = (tuple(v_flat), tuple(h_sampled))
        
        # 4. Increment the counter for this specific configuration
        config_counter[state_key] += 1
        
        # Advance the chain state
        v_current = v_next

    # 5. Estimate probabilities by dividing the occurrences by total steps [cite: 17]
    estimated_probabilities = {}
    for config, count in config_counter.items():
        estimated_probabilities[config] = count / steps
        
    # Print the top 5 most frequently visited states (empirical minimum energy states) [cite: 17, 18]
    print("\n--- [Task 1a] Simulation Results (Top 5 most frequent configurations): ---")
    most_common = config_counter.most_common(5)
    for rank, (config, count) in enumerate(most_common):
        v_state, h_state = config
        prob = estimated_probabilities[config]
        print(f"Rank {rank+1}: V={v_state}, H={h_state} | Occurrences: {count} | Estimated Probability: {prob:.6f}")
        
    return estimated_probabilities


# ==========================================
# Execution Example
# ==========================================
# Assuming 'rbm' was previously initialized and configured as your template specified:
# exact_results = measure_exact_energy_and_probabilities(rbm, m=5, n=3)
# sim_results = measure_gibbs_simulation(rbm, m=5, steps=100000)

In [ ]:
exact_results = measure_exact_energy_and_probabilities(rbm, m=5, n=3)

In [ ]:
sim_results = measure_gibbs_simulation(rbm, m=5, steps=100000)

In [ ]:
generate_all_configurations(m, n)[1]